# OpenThai 2.0 resolution sweep: raw versus gray220

Full 21-page OCR output is retained in this notebook for every run. The matrix is six longest-edge settings (`1024`, `1400`, `1600`, `1800`, `2000`, `2200`) crossed with original raw images and `gray220` images. Per-page preprocessing audit, GPU snapshots, benchmark configuration, timing, and Golden scores are captured as cell output.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import time
import urllib.request

ROOT = Path(".")
AUDIT_PYTHON = Path("python")
GOLDEN = ROOT / "results/bot_credit_bureau_21p_golden_transcript_20260829.json"
SWEEP_ROOT = ROOT / "inputs/bot_credit_bureau_2559/resolution_sweep"
MANIFEST = SWEEP_ROOT / "gray220_resolution_sweep_audit.json"
MODEL = "openthai2-qwen3.8-original-bf16"
ENDPOINT = "http://127.0.0.1:8097"
PROMPT_PROFILE = "document"
CONCURRENCY = 7
SERVER_MAX_NUM_SEQS = 7
MTP_TOKENS = 3
DISABLE_THINKING = True
SIZES = (1024, 1400, 1600, 1800, 2000, 2200)
TEMPERATURE = 0.0
TOP_P = 0.8
TOP_K = 20
REPETITION_PENALTY = 1.05
PRESENCE_PENALTY = 0.0
MAX_TOKENS = 8192

def gpu_snapshot():
    return subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()

def wait_for_model(timeout_seconds=900):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(ENDPOINT + "/v1/models", timeout=5) as response:
                payload = json.load(response)
            print("Endpoint ready:", ", ".join(item["id"] for item in payload.get("data", [])))
            return
        except Exception as error:
            last_error = error
            time.sleep(5)
    raise TimeoutError(f"Endpoint did not become ready: {last_error}")

def run_ocr(label, image_directory, result_path):
    output = ROOT / result_path
    if output.exists():
        raise FileExistsError(f"Refusing to overwrite {output}")
    command = [
        str(AUDIT_PYTHON), "benchmark/ocr_benchmark.py", str(ROOT / image_directory), str(output),
        "--endpoint", ENDPOINT,
        "--model", MODEL,
        "--prompt-profile", PROMPT_PROFILE,
        "--temperature", str(TEMPERATURE),
        "--top-p", str(TOP_P),
        "--top-k", str(TOP_K),
        "--repetition-penalty", str(REPETITION_PENALTY),
        "--presence-penalty", str(PRESENCE_PENALTY),
        "--max-tokens", str(MAX_TOKENS),
        "--concurrency", str(CONCURRENCY),
        "--images-per-request", "1",
        "--server-max-num-seqs", str(SERVER_MAX_NUM_SEQS),
    ]
    if DISABLE_THINKING:
        command.append("--disable-thinking")
    print(f"\n=== {label} ===")
    print("GPU before:\n" + gpu_snapshot())
    print("Command:", " ".join(command))
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=False)
    print(completed.stdout)
    if completed.stderr:
        print("STDERR:\n" + completed.stderr)
    print("GPU after:\n" + gpu_snapshot())
    if completed.returncode:
        raise RuntimeError(f"OCR exited with {completed.returncode}")
    return output

def show_all_pages(result_path):
    payload = json.loads(Path(result_path).read_text(encoding="utf-8"))
    print(json.dumps({"configuration": payload["configuration"], "summary": payload["summary"]}, ensure_ascii=False, indent=2))
    records = sorted(payload["records"], key=lambda item: int(Path(item["images"][0]).stem.split("-")[-1]))
    for record in records:
        page = int(Path(record["images"][0]).stem.split("-")[-1])
        print(f"\n--- OCR output: page {page:02d} ({record['elapsed_seconds']}s) ---")
        print(record.get("text", ""))

def show_gray_audit(longest_edge):
    payload = json.loads(MANIFEST.read_text(encoding="utf-8"))
    records = payload["sizes"][str(longest_edge)]["records"]
    print(json.dumps(records, ensure_ascii=False, indent=2))


In [2]:
sys.path.insert(0, str(ROOT / "benchmark"))
from ocr_benchmark import PROMPTS

print("Prompt profile:", PROMPT_PROFILE)
print(PROMPTS[PROMPT_PROFILE])
print("\nRun controls:")
print(json.dumps({
    "model": MODEL,
    "endpoint": ENDPOINT,
    "MTP_tokens": MTP_TOKENS,
    "server_max_num_seqs": SERVER_MAX_NUM_SEQS,
    "concurrency": CONCURRENCY,
    "disable_thinking": DISABLE_THINKING,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
}, ensure_ascii=False, indent=2))
wait_for_model()


Prompt profile: document
อ่านข้อความทุกส่วนของหน้ากระดาษนี้อย่างถูกต้องที่สุด และส่งคืนเฉพาะ Markdown ที่สกัดได้

กติกา:
- เก็บข้อความภาษาไทย อังกฤษ ตัวเลขอารบิก และตัวเลขไทยตามภาพ ห้ามแก้ไขหรือแปลงเลขไทยเป็นเลขอารบิก
- เก็บลำดับการอ่าน หัวข้อ รายการย่อย หัว/ท้ายกระดาษ และเชิงอรรถที่มองเห็น
- ตารางให้ส่งเป็น HTML <table> โดยคงแถว คอลัมน์ และหัวตาราง; ห้ามสรุปหรือข้ามเซลล์
- กราฟ แผนภาพ workflow และรูป ให้ห่อด้วย <figure> พร้อมถอดข้อความทุกคำก่อน แล้วบรรยายองค์ประกอบเท่าที่เห็น
- เมื่ออ่านข้อความไม่ชัด ให้เขียน [อ่านไม่ชัด] เฉพาะตำแหน่งนั้น ห้ามเดาหรือใช้ความรู้นอกภาพ
- ห้ามอธิบายวิธีทำ ห้ามเพิ่มข้อมูลที่ไม่อยู่ในภาพ

Run controls:
{
  "model": "openthai2-qwen3.8-original-bf16",
  "endpoint": "http://127.0.0.1:8097",
  "MTP_tokens": 3,
  "server_max_num_seqs": 7,
  "concurrency": 7,
  "disable_thinking": true,
  "temperature": 0.0,
  "max_tokens": 8192
}
Endpoint ready: openthai2-qwen3.8-original-bf16


## 1024px raw

In [3]:
raw_1024 = run_ocr("raw 1024px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024", "results/openthai2_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json")
show_all_pages(raw_1024)



=== raw 1024px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 89520
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024 ./results/openthai2_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8097 --model openthai2-qwen3.8-original-bf16 --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 14/21 (pages 014)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 19/21 (pages 019)
completed request 20/21 (pages 020)
completed request 1/21 (pages 001)
completed request 17/21 (pages 017)
completed request 21/21 (pages 021)
completed request 10/21 (pages 010)
{"images": 21, "requests": 21, "elapsed_seconds": 231.9419, "seconds_per_image": 11.0449, "completion_tokens": 49051, "end_to_end_completion_tokens_per_second": 211.48}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 90459
{
  "

## 1024px gray220

In [4]:
show_gray_audit(1024)
gray_1024 = run_ocr("gray220 1024px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024", "results/openthai2_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json")
show_all_pages(gray_1024)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024/page-01.png",
    "dimensions": [
      724,
      1024
    ],
    "threshold": 220,
    "pixels_replaced": 701722,
    "pixels_total": 741376,
    "pixels_replaced_percent": 94.6513,
    "raw_sha256": "52d5568aaef582dcec78889ffe4cf949683dd20c55c73ad0efb135690563596e",
    "gray220_sha256": "361f82d8dcccbf24b406e0de1d588675d31b4ba41964aeb8dddd18dc65ee73a3"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024/page-02.png",
    "dimensions": [
      724,
      1024
    ],
    "threshold": 220,
    "pi

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 6/21 (pages 006)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 11/21 (pages 011)
completed request 10/21 (pages 010)
completed request 14/21 (pages 014)
completed request 13/21 (pages 013)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 90.1279, "seconds_per_image": 4.2918, "completion_tokens": 18977, "end_to_end_completion_tokens_per_second": 210.556}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 91130
{
  "c

## 1400px raw

In [5]:
raw_1400 = run_ocr("raw 1400px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400", "results/openthai2_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json")
show_all_pages(raw_1400)



=== raw 1400px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 91130
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400 ./results/openthai2_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8097 --model openthai2-qwen3.8-original-bf16 --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 14/21 (pages 014)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
completed request 17/21 (pages 017)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 196.1909, "seconds_per_image": 9.3424, "completion_tokens": 27302, "end_to_end_completion_tokens_per_second": 139.16}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92143
{
  "c

## 1400px gray220

In [6]:
show_gray_audit(1400)
gray_1400 = run_ocr("gray220 1400px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400", "results/openthai2_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json")
show_all_pages(gray_1400)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400/page-01.png",
    "dimensions": [
      990,
      1400
    ],
    "threshold": 220,
    "pixels_replaced": 1323612,
    "pixels_total": 1386000,
    "pixels_replaced_percent": 95.4987,
    "raw_sha256": "027ea1c68561df52ce2beea388ee8c612768006d6feb1f4fa09bcabcff165a18",
    "gray220_sha256": "48c9be1e4a099ded0fdef40c00daade1db540a7efd6315a691070fb821bf077c"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400/page-02.png",
    "dimensions": [
      990,
      1400
    ],
    "threshold": 220,
    "

completed request 5/21 (pages 005)
completed request 1/21 (pages 001)
completed request 4/21 (pages 004)
completed request 2/21 (pages 002)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 14/21 (pages 014)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 15/21 (pages 015)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 84.9422, "seconds_per_image": 4.0449, "completion_tokens": 19181, "end_to_end_completion_tokens_per_second": 225.812}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92053
{
  "c

## 1600px raw

In [7]:
raw_1600 = run_ocr("raw 1600px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600", "results/openthai2_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json")
show_all_pages(raw_1600)



=== raw 1600px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92053
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600 ./results/openthai2_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8097 --model openthai2-qwen3.8-original-bf16 --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 11/21 (pages 011)
completed request 14/21 (pages 014)
completed request 13/21 (pages 013)
completed request 12/21 (pages 012)
completed request 18/21 (pages 018)
completed request 15/21 (pages 015)
completed request 16/21 (pages 016)
completed request 17/21 (pages 017)
completed request 20/21 (pages 020)
completed request 19/21 (pages 019)
completed request 10/21 (pages 010)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 208.7601, "seconds_per_image": 9.941, "completion_tokens": 34270, "end_to_end_completion_tokens_per_second": 164.16}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92802
{
  "co

## 1600px gray220

In [8]:
show_gray_audit(1600)
gray_1600 = run_ocr("gray220 1600px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600", "results/openthai2_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json")
show_all_pages(gray_1600)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600/page-01.png",
    "dimensions": [
      1131,
      1600
    ],
    "threshold": 220,
    "pixels_replaced": 1732917,
    "pixels_total": 1809600,
    "pixels_replaced_percent": 95.7624,
    "raw_sha256": "7cd88e5cf12e59d83931179971c152df49e476e3e08d7ba1eb6be1fa0f5d46b7",
    "gray220_sha256": "eaf2a4fa6c8aa8ea8200b12cd7eaaa81dd4aa418aab469cf58f59cbc1d5c13e2"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600/page-02.png",
    "dimensions": [
      1131,
      1600
    ],
    "threshold": 220,
   

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 2/21 (pages 002)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 11/21 (pages 011)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 85.4609, "seconds_per_image": 4.0696, "completion_tokens": 18900, "end_to_end_completion_tokens_per_second": 221.154}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 93026
{
  "c

## 1800px raw

In [9]:
raw_1800 = run_ocr("raw 1800px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800", "results/openthai2_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json")
show_all_pages(raw_1800)



=== raw 1800px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 93026
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800 ./results/openthai2_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8097 --model openthai2-qwen3.8-original-bf16 --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 11/21 (pages 011)
completed request 10/21 (pages 010)
completed request 14/21 (pages 014)
completed request 13/21 (pages 013)
completed request 12/21 (pages 012)
completed request 18/21 (pages 018)
completed request 21/21 (pages 021)
completed request 15/21 (pages 015)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 86.056, "seconds_per_image": 4.0979, "completion_tokens": 19804, "end_to_end_completion_tokens_per_second": 230.129}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92974
{
  "co

## 1800px gray220

In [10]:
show_gray_audit(1800)
gray_1800 = run_ocr("gray220 1800px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800", "results/openthai2_bot_credit_bureau_21p_gray220_px1800_resolution_sweep_20260829.json")
show_all_pages(gray_1800)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800/page-01.png",
    "dimensions": [
      1272,
      1800
    ],
    "threshold": 220,
    "pixels_replaced": 2197434,
    "pixels_total": 2289600,
    "pixels_replaced_percent": 95.9746,
    "raw_sha256": "ea45e6faa8ac8d1bc413f017d0428dae7458faee05babfde03c571c008faf256",
    "gray220_sha256": "e9aaa8c9c0ebf36bd4f65e716f6e8ebe06fa133de12a32db5f5af0df5df13234"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800/page-02.png",
    "dimensions": [
      1272,
      1800
    ],
    "threshold": 220,
   

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 13/21 (pages 013)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 83.7375, "seconds_per_image": 3.9875, "completion_tokens": 18735, "end_to_end_completion_tokens_per_second": 223.735}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92773
{
  "c

## 2000px raw

In [11]:
raw_2000 = run_ocr("raw 2000px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000", "results/openthai2_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json")
show_all_pages(raw_2000)



=== raw 2000px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92777
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000 ./results/openthai2_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8097 --model openthai2-qwen3.8-original-bf16 --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 14/21 (pages 014)
completed request 13/21 (pages 013)
completed request 12/21 (pages 012)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
completed request 21/21 (pages 021)
{"images": 21, "requests": 21, "elapsed_seconds": 172.8607, "seconds_per_image": 8.2315, "completion_tokens": 27088, "end_to_end_completion_tokens_per_second": 156.704}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92782
{
  "

## 2000px gray220

In [12]:
show_gray_audit(2000)
gray_2000 = run_ocr("gray220 2000px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000", "results/openthai2_bot_credit_bureau_21p_gray220_px2000_resolution_sweep_20260829.json")
show_all_pages(gray_2000)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000/page-01.png",
    "dimensions": [
      1414,
      2000
    ],
    "threshold": 220,
    "pixels_replaced": 2718321,
    "pixels_total": 2828000,
    "pixels_replaced_percent": 96.1217,
    "raw_sha256": "d4a80b77100cb787878ee295a47b493e6c6afbe6b6619cf11bad88dc30e67a46",
    "gray220_sha256": "ee2902a15cdec3354762e8361c56bb13eb98bf8e0675d74bf0f719754b0f9ae3"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000/page-02.png",
    "dimensions": [
      1414,
      2000
    ],
    "threshold": 220,
   

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 11/21 (pages 011)
completed request 10/21 (pages 010)
completed request 14/21 (pages 014)
completed request 13/21 (pages 013)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 85.9624, "seconds_per_image": 4.0934, "completion_tokens": 18775, "end_to_end_completion_tokens_per_second": 218.41}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92773
{
  "co

## 2200px raw

In [13]:
raw_2200 = run_ocr("raw 2200px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200", "results/openthai2_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json")
show_all_pages(raw_2200)



=== raw 2200px ===
GPU before:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92773
Command: python benchmark/ocr_benchmark.py ./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200 ./results/openthai2_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json --endpoint http://127.0.0.1:8097 --model openthai2-qwen3.8-original-bf16 --prompt-profile document --temperature 0.0 --top-p 0.8 --top-k 20 --repetition-penalty 1.05 --presence-penalty 0.0 --max-tokens 8192 --concurrency 7 --images-per-request 1 --server-max-num-seqs 7 --disable-thinking


completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 2/21 (pages 002)
completed request 3/21 (pages 003)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 13/21 (pages 013)
completed request 11/21 (pages 011)
completed request 14/21 (pages 014)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 15/21 (pages 015)
completed request 18/21 (pages 018)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 90.3431, "seconds_per_image": 4.3021, "completion_tokens": 19452, "end_to_end_completion_tokens_per_second": 215.313}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92842
{
  "c

## 2200px gray220

In [14]:
show_gray_audit(2200)
gray_2200 = run_ocr("gray220 2200px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200", "results/openthai2_bot_credit_bureau_21p_gray220_px2200_resolution_sweep_20260829.json")
show_all_pages(gray_2200)


[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200/page-01.png",
    "dimensions": [
      1555,
      2200
    ],
    "threshold": 220,
    "pixels_replaced": 3302179,
    "pixels_total": 3421000,
    "pixels_replaced_percent": 96.5267,
    "raw_sha256": "289cc186f8272818eebe04ab7817c751400dc99e89e853689e300a2a35c950b1",
    "gray220_sha256": "aca77e3bbbc84f3448de60b79c196caa439ce1ccdefeb9725d48733840f59327"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200/page-02.png",
    "dimensions": [
      1555,
      2200
    ],
    "threshold": 220,
   

completed request 1/21 (pages 001)
completed request 5/21 (pages 005)
completed request 4/21 (pages 004)
completed request 3/21 (pages 003)
completed request 2/21 (pages 002)
completed request 6/21 (pages 006)
completed request 7/21 (pages 007)
completed request 8/21 (pages 008)
completed request 9/21 (pages 009)
completed request 10/21 (pages 010)
completed request 11/21 (pages 011)
completed request 14/21 (pages 014)
completed request 13/21 (pages 013)
completed request 12/21 (pages 012)
completed request 21/21 (pages 021)
completed request 18/21 (pages 018)
completed request 15/21 (pages 015)
completed request 16/21 (pages 016)
completed request 20/21 (pages 020)
completed request 17/21 (pages 017)
completed request 19/21 (pages 019)
{"images": 21, "requests": 21, "elapsed_seconds": 87.5161, "seconds_per_image": 4.1674, "completion_tokens": 18663, "end_to_end_completion_tokens_per_second": 213.252}

GPU after:
0, NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887, 92806
{
  "c

## Golden score for every size and preprocessing variant

In [15]:
score_path = ROOT / "results/openthai2_resolution_sweep_raw_gray220_20260829.score.json"
score_command = [str(AUDIT_PYTHON), "benchmark/score_golden_ocr.py", str(GOLDEN), str(score_path), "--target", "raw_px1024=results/openthai2_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json", "--target", "gray220_px1024=results/openthai2_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json", "--target", "raw_px1400=results/openthai2_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json", "--target", "gray220_px1400=results/openthai2_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json", "--target", "raw_px1600=results/openthai2_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json", "--target", "gray220_px1600=results/openthai2_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json", "--target", "raw_px1800=results/openthai2_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json", "--target", "gray220_px1800=results/openthai2_bot_credit_bureau_21p_gray220_px1800_resolution_sweep_20260829.json", "--target", "raw_px2000=results/openthai2_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json", "--target", "gray220_px2000=results/openthai2_bot_credit_bureau_21p_gray220_px2000_resolution_sweep_20260829.json", "--target", "raw_px2200=results/openthai2_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json", "--target", "gray220_px2200=results/openthai2_bot_credit_bureau_21p_gray220_px2200_resolution_sweep_20260829.json"]
print("Command:", " ".join(score_command))
completed = subprocess.run(score_command, cwd=ROOT, text=True, capture_output=True, check=True)
print(completed.stdout)
if completed.stderr:
    print("STDERR:\n" + completed.stderr)
score_payload = json.loads(score_path.read_text(encoding="utf-8"))
for label, target in score_payload["targets"].items():
    summary = target["summary"]
    print("\n", label)
    print(json.dumps({
        "word_accuracy_percent": summary["words"]["accuracy_percent"],
        "word_deltas": {key: summary["words"][key] for key in ("substituted", "missing", "extra")},
        "number_accuracy_percent": summary["number_tokens"]["accuracy_percent"],
    }, ensure_ascii=False, indent=2))


Command: python benchmark/score_golden_ocr.py ./results/bot_credit_bureau_21p_golden_transcript_20260829.json ./results/openthai2_resolution_sweep_raw_gray220_20260829.score.json --target raw_px1024=results/openthai2_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json --target gray220_px1024=results/openthai2_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json --target raw_px1400=results/openthai2_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json --target gray220_px1400=results/openthai2_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json --target raw_px1600=results/openthai2_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json --target gray220_px1600=results/openthai2_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json --target raw_px1800=results/openthai2_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json --target gray220_px1800=results/openthai2_bot_credit_bureau_21p_gray220_px1800_reso

{"raw_px1024": {"scored_pages": 21, "words": {"correct": 5198, "substituted": 880, "missing": 787, "extra": 6180, "golden_count": 6865, "ocr_count": 12258, "accuracy_percent": 75.717}, "number_tokens": {"correct": 165, "substituted": 65, "missing": 134, "extra": 836, "accuracy_percent": 45.33}}, "gray220_px1024": {"scored_pages": 21, "words": {"correct": 5737, "substituted": 671, "missing": 457, "extra": 338, "golden_count": 6865, "ocr_count": 6746, "accuracy_percent": 83.569}, "number_tokens": {"correct": 185, "substituted": 46, "missing": 133, "extra": 4, "accuracy_percent": 50.824}}, "raw_px1400": {"scored_pages": 21, "words": {"correct": 6230, "substituted": 323, "missing": 312, "extra": 1065, "golden_count": 6865, "ocr_count": 7618, "accuracy_percent": 90.75}, "number_tokens": {"correct": 175, "substituted": 42, "missing": 147, "extra": 3, "accuracy_percent": 48.077}}, "gray220_px1400": {"scored_pages": 21, "words": {"correct": 6279, "substituted": 283, "missing": 303, "extra": 11